# NEURON: soma, dendrite, and spatial resolution

Use the **Science - NEURON** kernel. This is a software-verification and cable-model lesson, not a fitted human neuron or clinical model. The built-in `hh` mechanism is historically squid-axon-derived.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
from neuron import h

root = Path.cwd()
assert (root / 'verify_neuron.py').exists(), 'Open from the science-workbench folder.'
h.load_file('stdrun.hoc')
print(h.nrnversion())

## Prediction and model boundary

AI checkpoint prediction: a brief soma current will trigger an active spike; the passive dendritic tip will depolarize with attenuation and delay. Increasing spatial resolution while holding the time step fixed should change the trace slightly, not reverse that qualitative result. Replace this with your own prediction before changing parameters.

In [ ]:
def simulate(dend_nseg=21, dt=0.025):
    soma = h.Section(name=f'soma_{dend_nseg}')
    dend = h.Section(name=f'dend_{dend_nseg}')
    dend.connect(soma(1))

    soma.L = soma.diam = 20
    soma.nseg = 1
    soma.insert('hh')

    dend.L = 200
    dend.diam = 2
    dend.nseg = dend_nseg
    dend.insert('pas')
    dend.g_pas = 0.001
    dend.e_pas = -65

    stim = h.IClamp(soma(0.5))
    stim.delay = 5
    stim.dur = 1
    stim.amp = 0.5

    time = h.Vector().record(h._ref_t)
    soma_v = h.Vector().record(soma(0.5)._ref_v)
    middle_v = h.Vector().record(dend(0.5)._ref_v)
    tip_v = h.Vector().record(dend(1)._ref_v)

    h.dt = dt
    h.finitialize(-65)
    h.continuerun(20)
    return tuple(np.asarray(v) for v in (time, soma_v, middle_v, tip_v))

baseline = simulate(dend_nseg=21, dt=0.025)
finer = simulate(dend_nseg=41, dt=0.025)
len(baseline[0]), len(finer[0])

In [ ]:
time, soma_v, middle_v, tip_v = baseline
finer_tip_on_baseline = np.interp(time, finer[0], finer[3])
metrics = {
    'samples': int(len(time)),
    'soma_peak_mv': float(soma_v.max()),
    'middle_peak_mv': float(middle_v.max()),
    'tip_peak_mv': float(tip_v.max()),
    'peak_attenuation_mv': float(soma_v.max() - tip_v.max()),
    'finer_tip_max_abs_difference_mv': float(np.max(np.abs(tip_v - finer_tip_on_baseline))),
}
checks = {
    'active_soma_spiked': metrics['soma_peak_mv'] > 0,
    'dendritic_tip_depolarized': metrics['tip_peak_mv'] > -40,
    'tip_peak_below_soma_peak': metrics['tip_peak_mv'] < metrics['soma_peak_mv'],
    'resolution_difference_below_1_mv': metrics['finer_tip_max_abs_difference_mv'] < 1,
}
assert all(checks.values()), (metrics, checks)
metrics, checks

In [ ]:
result_dir = root / 'results' / 'neuron_compartments'
result_dir.mkdir(parents=True, exist_ok=True)
np.savetxt(result_dir / 'baseline_voltage.csv', np.column_stack(baseline), delimiter=',', header='time_ms,soma_mv,middle_mv,tip_mv', comments='')
(result_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2) + '\n', encoding='ascii')
(result_dir / 'checks.json').write_text(json.dumps(checks, indent=2) + '\n', encoding='ascii')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(time, soma_v, label='soma (active HH)')
ax.plot(time, middle_v, label='dendrite midpoint (passive)')
ax.plot(time, tip_v, label='dendrite tip (passive)')
ax.set(xlabel='Time (ms)', ylabel='Voltage (mV)', title='Voltage propagation along a 200 um dendrite')
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(result_dir / 'voltage_propagation.png', dpi=160)
plt.show()

## Interpret before extending

Explain why the soma and dendritic tip differ even though they are connected. Then identify which claims are numerical (the resolution comparison), mechanistic (active soma versus passive dendrite), and out of scope (human cell identity, learning, mental health, or drug response).

Next experiment: write a prediction, copy the fixed baseline, then change exactly one of dendrite length, diameter, passive conductance, or stimulus amplitude.